***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [1. 利用干涉阵开展射电科学](1_0_introduction.ipynb)
    * 上一节： [1.8 天文射电源](1_8_astronomical_radio_sources.ipynb)
    * 下一节： [1.10 单碟望远镜的局限与价值](1_10_limits_of_single_dishes.ipynb)
***


导入标准模块:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML 
HTML('../style/course.css') #apply general CSS

导入本节所需的专用模块:

In [ ]:
from IPython.display import display
try:
    from ipywidgets import interact
except ImportError:
    def interact(func, **kwargs):
        defaults = {}
        for key, value in kwargs.items():
            if isinstance(value, tuple):
                defaults[key] = value[0]
            else:
                defaults[key] = value
        return func(**defaults)


In [ ]:
HTML('../style/code_toggle.html')

## 1.9 干涉测量：从双缝到孔径合成

前述辐射机制和射电源对角分辨率、空间尺度响应和测量精度提出了不同要求。干涉阵通过比较分散阵元接收到的电场信号，获得单个口径无法提供的空间信息。本节从双缝实验出发，说明相位差、基线和源结构之间的基本关系，并进一步引出复可见度与孔径合成。

本节建立三个基本概念：干涉测量比较不同路径或不同阵元信号之间的相位关系；单条基线只提供天空亮度分布的一个复数约束；多条基线与地球自转共同增加空间频率采样，使天空亮度分布的重建成为可能。这些概念构成后续可见度、$uv$ 采样和成像反演的物理基础。


### 1.9.1 双缝实验中的相位与强度

干涉测量的历史基础可追溯到 1801 年托马斯·杨提出的双缝实验。该实验最初用于证明光的波动性，同时清楚展示了相位差如何转化为可测量的强度变化，因此也为理解射电干涉测量提供了简明的物理模型。

<img src="figures/double_slit_schematic.png" width="80%"/>

**图 1.9.1**：双缝实验的路径差与屏上强度条纹。图为本项目生成的概念示意，不按实验装置比例绘制。

在双缝实验中，两条路径上的电场在屏幕处相加。若两路单色波写成复振幅形式，则有

$$E = E_1 + E_2 = A e^{i\phi} + A e^{i(\phi-\phi_0)}$$

其中 $\phi_0$ 表示两条路径产生的相位差。探测器记录的是时间平均强度，而非瞬时电场；强度与 $EE^*$ 成正比。展开后可得

$$EE^* = 2A^2 + 2A^2\cos\phi_0$$

干涉条纹表示相位差转化形成的强度起伏。现代射电干涉仪不直接记录屏幕条纹，而是测量不同阵元接收电场之间的互相关及其相位关系。不同天体或不同微观发射区产生的随机辐射通常彼此不相干，其贡献在统计平均意义下相加；干涉仪利用的是同一天空辐射场在不同阵元处的相关性，而不是独立天体之间的相干叠加。


### 1.9.2 双缝干涉的简化模拟

下面的 Python 函数采用一维远场近似，用于显示路径差对条纹相位的影响以及源结构对条纹对比度的影响。该模型未包含完整的衍射传播和真实光学系统响应，仅用于说明基本关系。


In [ ]:
def double_slit(p0=[0], a0=[1], baseline=1, d1=5, d2=5, wavelength=.1, maxint=None):
    """绘制简化的双缝干涉模型。
    p0：沿纵轴给出的源位置列表或数组。
    a0：各源的强度数组。
    baseline：双缝间距。
    d1、d2：源到狭缝平面、狭缝平面到像屏的距离。
    wavelength：波长。
    maxint：条纹图样的最大强度标度；为 None 时自动缩放。设置固定值可在统一强度标度下比较多次调用结果。
    """
    # 设置绘图区与坐标轴
    plt.figure(figsize=(20, 5))
    plt.axes(frameon=False)
    plt.xlim(-d1-.1, d2+2) and plt.ylim(-1, 1)
    plt.xticks([]) and plt.yticks([])
    plt.axhline(0, ls=':')
    baseline /= 2.
    # 绘制双缝
    plt.arrow(0, 1,0, baseline-1, lw=0, width=.1, head_width=.1, length_includes_head=True)
    plt.arrow(0,-1,0, 1-baseline, lw=0, width=.1, head_width=.1, length_includes_head=True)
    plt.arrow(0, 0,0,  baseline,  lw=0, width=.1, head_width=.1, length_includes_head=True)
    plt.arrow(0, 0,0, -baseline,  lw=0, width=.1, head_width=.1, length_includes_head=True)
    # 绘制双缝到像屏中心的传播路径
    plt.arrow(0, baseline,d2,-baseline, length_includes_head=True)
    plt.arrow(0,-baseline,d2, baseline, length_includes_head=True)
    # 绘制中心位置的正弦波示意
    xw = np.arange(-d1, -d1+(d1+d2)/4, .01)
    yw = np.sin(2*np.pi*xw/wavelength)*.1 + (p0[0]+p0[-1])/2
    plt.plot(xw,yw,'b')
    # xs 表示像屏上的纵向坐标；pattern 累加各源的干涉图样
    xs = np.arange(-1, 1, .01) 
    pattern = 0
    total_intensity = 0
    # 计算各源位置对干涉图样的贡献
    for p,a in np.broadcast(p0,a0):
        plt.plot(-d1, p, marker='o', ms=10, mfc='red', mew=0)
        total_intensity += a
        if p == p0[0] or p == p0[-1]:
            plt.arrow(-d1, p, d1, baseline-p, length_includes_head=True)
            plt.arrow(-d1, p, d1,-baseline-p, length_includes_head=True)
        # 计算两条传播路径的长度
        path1 = np.sqrt(d1**2 + (p-baseline)**2) + np.sqrt(d2**2 + (xs-baseline)**2)
        path2 = np.sqrt(d1**2 + (p+baseline)**2) + np.sqrt(d2**2 + (xs+baseline)**2)
        diff = path1 - path2
        # 累加当前源的干涉图样
        pattern = pattern + a*np.cos(2*np.pi*diff/wavelength) 
    maxint = maxint or total_intensity
    # 扩展为二维数组，以条带形式显示干涉图样
    pattern_image = pattern[:,np.newaxis] + np.zeros(10)[np.newaxis,:]
    plt.imshow(pattern_image, extent=(d2,d2+1,-1,1), cmap=plt.gray(), vmin=-maxint, vmax=maxint)
    # 绘制干涉图样的强度截面
    plt.plot(d2+1.5+pattern/(maxint*2), xs, 'r')
    plt.show()
# 显示位于中心位置的单源干涉图样
double_slit(p0=[0])

`double_slit` 函数绘制简化的双缝装置。红点表示源位置，蓝色正弦曲线表示波长，黑线表示传播路径，右侧灰度条带和红色曲线分别表示条纹图样及其截面。该模型用于比较条纹随基线、波长和源结构的变化规律。

<div class=warn>
<b>注意：</b> 该模型采用一维远场近似，未严格处理衍射传播和真实光学系统。图中细节不能直接用于描述实际干涉阵的工程响应。
</div>


### 1.9.3 基线、波长与角分辨率

对于点源，增大基线 $B$ 或减小波长 $\lambda$ 都会使条纹间距减小，表明系统对角位置变化更加敏感。干涉测量的特征角分辨率因此按 $\lambda/B$ 缩放。


In [ ]:
interact(lambda baseline,wavelength:double_slit(p0=[0],baseline=baseline,wavelength=wavelength),
                baseline=(0.1,2,.01),wavelength=(.05,.2,.01)) and None

条纹间距减小不等于已经获得更清晰的图像，而是表示系统对源位置和结构变化更加敏感。基线决定干涉仪对特定空间尺度的响应，不直接决定图像像素大小。形成二维图像还需要多条基线提供的空间频率采样以及相应的重建过程。


### 1.9.4 从双缝实验到可测量的天文信息

将双缝装置视为测量系统时，条纹相位和对比度可分别用于约束源的位置与结构。这一对应关系是天文干涉测量的基本出发点。

#### 1.9.4.1 位置测量：可见度相位与源位置

保持点源结构不变，仅改变其相对于光轴的位置时，干涉图样的整体相位随之变化，且长基线对这种位置变化更为敏感。


In [ ]:
interact(lambda position,baseline,wavelength:double_slit(p0=[position],baseline=baseline,wavelength=wavelength),
               position=(-1,1,.01),baseline=(0.1,2,.01),wavelength=(.05,.2,.01)) and None

在最简单的点源情形下，可见度相位直接编码源位置，并表现为随基线变化的复相位因子。长基线对位置变化更敏感，因而具有较高角分辨能力；但相位的周期性也会使单基线位置测量产生多解。


In [ ]:
double_slit([0],baseline=1.5,wavelength=0.1)
double_slit([0.69],baseline=1.5,wavelength=0.1)

上面两组位置完全不同的源，在同一条长基线上却可能给出几乎相同的干涉图样。这说明一条基线本身并不能唯一确定源的位置，尤其当基线很长时，条纹周期性会引入模糊性。


In [ ]:
double_slit([0],baseline=0.5,wavelength=0.1)
double_slit([0.69],baseline=0.5,wavelength=0.1)

较短基线虽然角分辨率较低，却有助于消除长基线相位周期性造成的位置歧义。因此，现代干涉阵需要同时包含不同长度和方向的基线。对每个时间、频率通道和偏振相关乘积，单条基线只产生一个复可见度样本，而非一幅图像。时间采样、频率采样和多偏振相关可增加样本数量，但天空结构的恢复仍依赖多基线信息的联合约束。


#### 1.9.4.2 尺度测量：可见度振幅与源尺度

相位主要反映位置，干涉图样的对比度或可见度振幅则更直接地反映源结构。下面加入第二个点源，以比较双点源结构对干涉图样的影响。


In [ ]:
interact(lambda position,intensity,baseline,wavelength:
            double_slit(p0=[0,position],a0=[1,intensity],baseline=baseline,wavelength=wavelength),
         position=(-1,1,.01),intensity=(.2,1,.01),baseline=(0.1,2,.01),wavelength=(.01,.2,.01)) and None

不同的位置和强度组合会降低条纹对比度，并可能在某些基线上产生近似完全抵消。可见度振幅因此对源结构敏感。对于双点源，特定基线上的条纹对比度反映该基线对两分量间角距离的响应。


In [ ]:
double_slit(p0=[0,0.25],baseline=1,wavelength=0.1)
double_slit(p0=[0,0.25],baseline=1.5,wavelength=0.1)

对于连续展源，不同位置辐射产生的相位贡献会部分相消，使条纹振幅随源尺度增大而降低。下面通过逐渐增加源宽度显示这一效应。


In [ ]:
interact(lambda extent,baseline,wavelength:
             double_slit(p0=np.arange(-extent,extent+.01,.01),baseline=baseline,wavelength=wavelength),
         extent=(0,1,.01),baseline=(0.1,2,.01),wavelength=(.01,.2,.01)) and None

该结果表明，长基线对大尺度结构的可见度振幅较低，是因为这类结构在相应空间频率上已被分辨，而不是因为结构不存在。不同基线长度对应不同空间尺度的响应，不能将其理解为对同一图像不同像素的直接测量。


In [ ]:
double_slit(p0=[0],baseline=1,wavelength=0.1)
double_slit(p0=np.arange(-0.2,.21,.01),baseline=1,wavelength=0.1)

在最简单的两阵元干涉仪中，一条基线的测量结果可表示为复可见度

$$V(B)=|V(B)|e^{i\phi(B)}$$

其中 $|V|$ 近似反映源在这条基线对应尺度上的结构响应，$\phi$ 近似反映相对于参考方向的位置偏移。后面第 4 章会把这一点推广到二维基线和完整天空亮度分布，并给出严格的傅里叶关系。

“可见度”一词源于历史上的条纹可见度，即无量纲对比度 $\mathcal{V}=(I_{\max}-I_{\min})/(I_{\max}+I_{\min})$。现代射电干涉测量中，经过通量标定的复可见度通常保留相关通量密度的量纲（常用 Jy），并同时包含振幅和相位；使用总强度或自相关归一化后，才得到接近无量纲相干度的量。因此，数值为 0.5 的条纹对比度与 $0.5$ Jy 的可见度振幅表示不同的物理量。


#### 1.9.4.3 仪器几何对干涉图样的影响

干涉图样不仅取决于源结构，也取决于仪器几何。狭缝位置、屏幕位置和光路长度的变化都会改变条纹形态。在实际干涉测量中，阵元位置、时延和系统响应的不确定性会使观测相位与振幅同时包含天体信号和仪器效应，因此必须通过校准加以分离。


In [ ]:
interact(lambda d1,d2,position,extent: double_slit(p0=np.arange(position-extent,position+extent+.01,.01),d1=d1,d2=d2),
         d1=(1,5,.1),d2=(1,5,.1),
         position=(-1,1,.01),extent=(0,1,.01)) and None

相位对几何变化的高灵敏度也可用于精密测量。大地测量 VLBI 利用已知射电源约束站间基线和地壳运动，激光干涉引力波探测器则利用相位变化测量微小光程差。二者均体现了干涉相位对几何参数的精密约束能力。


### 1.9.5 选读：天文干涉仪的实现方式

双缝实验说明了相位差向可测信号的转换，而天文干涉仪还需处理远场波前、长基线和随时间变化的几何时延。其主要实现方式可分为两类：

| 实现方式 | 信号组合方式 | 典型输出 | 关键要求 |
|:---|:---|:---|:---|
| 加法式干涉仪 | 两路电场先合束，再测总强度中的干涉项 | 条纹及其对比度 | 稳定并补偿真实光程差 |
| 乘法式干涉仪 | 各阵元独立接收、放大和数字化，再计算交叉相关 | 复可见度 $\langle E_iE_j^*\rangle$ | 时钟、延迟模型、频率与相位标定 |

迈克尔逊恒星干涉仪属于前一类；1920 年 Michelson 和 Pease 对参宿四角直径的测量，是用基线改变条纹可见度来约束源尺寸的经典例子。早期射电海崖干涉仪则利用直达与海面反射形成两条路径。现代连接阵采用后一类路线，把每个阵元的电压流送入延迟、通道化和相关处理：

<img src="figures/connected_array_correlator.png" width="85%"/>

**图 1.9.2**：连接阵把各阵元的独立电压流送入延迟、通道化与相关处理，输出按天线对组织的复可见度。图为本项目生成。

若来自方向 $\hat{s}$ 的平面波在两天线间产生几何时延 $\tau_g=\mathbf{b}\cdot\hat{s}/c$，相关器必须用延迟模型把已知几何相位与天体结构信息分开。因此现代射电干涉仪天然输出复可见度，而不是条纹照片；校准则负责从这个复数测量中分离仪器增益、时钟、传播介质和几何误差。


### 1.9.6 孔径合成与多基线成像

早期干涉测量常用于恒星角直径、源位置或几何结构等定量测量。孔径合成将单基线测量扩展为成像方法：不同长度和方向的基线分别采样天空亮度分布的不同空间频率，联合这些样本可重建天空图像。

下面三种不同的天空亮度分布在某一条特定基线上可能产生相近响应，因而无法由该基线单独区分：


In [ ]:
double_slit(p0=[0], a0=[0.4], maxint=2)
double_slit(p0=[0,0.25], a0=[1, 0.6], maxint=2)
double_slit(p0=np.arange(-0.2,.21,.01), a0=.05, maxint=2)

更换基线长度或方向后，三种天空模型的响应将出现差异：


In [ ]:
double_slit(p0=[0], a0=[0.4], baseline=0.5, maxint=2)
double_slit(p0=[0,0.25], a0=[1, 0.6], baseline=0.5, maxint=2)
double_slit(p0=np.arange(-0.2,.21,.01), a0=.05,  baseline=0.5, maxint=2)

孔径合成利用不同基线采样不同空间尺度和方向的信息。按照第 4 章将采用的表述，每一条基线测量天空亮度分布的一个复数投影。对于小视场、主波束近似不变的标量情形，可写成二维关系

$$V(u,v) = \iint I(l,m)\,e^{-2\pi i(ul+vm)}\,dl\,dm.$$

其中 $(u,v)$ 是以波长为单位的基线坐标，$(l,m)$ 是天空方向余弦。宽视场中的 $w$ 项、$1/n$ 因子以及频率或方向相关主波束会改变这一简化关系；完整测量方程及近似条件将在第 4 章讨论。可见度不是图像本身，而是天空亮度分布的空间频率信息。

地球自转、较多阵元和多构型观测能够增加 $uv$ 采样，使干涉阵由单一复数投影测量扩展为天空亮度分布成像系统。实际采样始终有限，因此图像还会受到点扩散函数、权重选择、短间距缺失和噪声的影响。这些问题将在后续成像与去卷积章节中系统讨论。


### 1.9.7 本节小结

本节从双缝实验出发，建立了干涉测量的基本直觉：不同路径上的相位差可以转化成可测量的强度或相关信号。对天文干涉仪而言，一条基线在每个时间、频率和偏振样本上返回一个复可见度，而不是一张完整图像；在简单情形下，可见度相位主要编码源位置，可见度振幅主要反映源结构和尺度。历史条纹对比度是无量纲归一化量，现代标定复可见度则通常以 Jy 表示相关通量。

实际干涉测量同时受到仪器几何、时延和系统响应的影响，因此必须进行校准。孔径合成依靠多条基线对天空亮度分布不同空间频率分量的联合采样实现成像。下一节将在此基础上比较单碟望远镜与干涉阵的尺度响应、科学功能和技术代价。


### 附录（选读）：迈克尔逊恒星干涉仪的简化模拟

下面给出一个接近迈克尔逊恒星干涉仪布局的简化模型。模型采用无限远光源和迈克尔逊式光路，用于进一步说明条纹图样、可见度与有限基线长度之间的关系。本附录为选读内容，不构成本章主线的先决条件。


In [ ]:
def michelson(p0=[0], a0=[1], baseline=50, maxbaseline=100, extent=0, d1=9, d2=1, d3=.2, wavelength=.1, fov=5, maxint=None):
    """绘制无限远天文源的简化迈克尔逊干涉仪模型。
    p0：以度为单位的源位置列表或数组。
    a0：各源的强度数组。
    extent：以度为单位的源角尺度。
    baseline、maxbaseline：以波长为单位的基线长度及绘图最大基线。
    d1、d2、d3：绘图中天空、干涉臂、像屏及内侧反射镜之间的相对距离。
    fov：以度为单位的显示视场半径。
    wavelength：用于绘图标度的波长。
    maxint：条纹图样的最大强度标度；为 None 时自动缩放。设置固定值可在统一强度标度下比较多次调用结果。
    """
    # 设置绘图区与坐标轴
    plt.figure(figsize=(20, 5))
    plt.axes(frameon=False)
    plt.xlim(-d1-.1, d2+2) and plt.ylim(-1, 1)
    plt.xticks([]) 
    # 以度为单位标记纵轴
    yt,ytlab = plt.yticks()
    plt.yticks(yt,["-%g"%(float(y)*fov) for y in yt]) 
    plt.ylabel("到达角（度）")
    plt.axhline(0, ls=':')
    # 绘制干涉臂与传播路径
    maxbaseline = max(maxbaseline,baseline)
    bl2 = baseline/float(maxbaseline)    # 绘图坐标中的半基线长度
    plt.plot([0,0],[-bl2,bl2], 'o', ms=10)
    plt.plot([0,d2/2.,d2/2.,d2],[-bl2,-bl2,-d3/2.,0],'-k')
    plt.plot([0,d2/2.,d2/2.,d2],[ bl2, bl2, d3/2.,0],'-k')
    plt.text(0, 0, r'$b=%d\lambda$' % baseline, ha='right', va='bottom', size='xx-large')
    # 绘制中心位置的正弦波示意
    if isinstance(p0,(int,float)):
        p0 = [p0]
    xw = np.arange(-d1, -d1+(d1+d2)/4, .01)
    yw = np.sin(2*np.pi*xw/wavelength)*.1 + (p0[0]+p0[-1])/(2.*fov)
    plt.plot(xw,yw,'b')
    # xs 表示像屏上的纵向坐标
    xs = np.arange(-1, 1, .01) 
    # xsdiff 表示像屏位置对应的路径差
    xsdiff = (np.sqrt(d2**2 + (xs-d3)**2) - np.sqrt(d2**2 + (xs+d3)**2))
    # pattern 累加各源的干涉图样
    pattern = 0
    total_intensity = 0
    # 计算各源位置对干涉图样的贡献
    for pos,ampl in np.broadcast(p0,a0):
        total_intensity += ampl
        pos1 = pos/float(fov)
        if extent:  # 用一组等强度点源近似扩展源
            positions = np.arange(-1,1.01,.01)*extent/fov + pos1 
        else:
            positions = [pos1]
        # 绘制传播路径
        plt.arrow(-d1, bl2+pos1, d1, -pos1, head_width=.1, fc='k', length_includes_head=True)
        plt.arrow(-d1,-bl2+pos1, d1, -pos1, head_width=.1, fc='k', length_includes_head=True)
        for p in positions:
            # 计算两干涉臂到像屏位置的路径差
            plt.plot(-d1, p, marker='o', ms=10*ampl, mfc='red', mew=0)
            # 加入源到达角造成的路径差
            diff = xsdiff + (baseline*wavelength)*np.sin(p*fov*np.pi/180)
            # 累加当前源的干涉图样
            pattern = pattern + (float(ampl)/len(positions))*np.cos(2*np.pi*diff/wavelength) 
    maxint = maxint or total_intensity
    # 扩展为二维数组，以条带形式显示干涉图样
    pattern_image = pattern[:,np.newaxis] + np.zeros(10)[np.newaxis,:]
    plt.imshow(pattern_image, extent=(d2,d2+1,-1,1), cmap=plt.gray(), vmin=-maxint, vmax=maxint)
    # 绘制干涉图样的强度截面
    plt.plot(d2+1.5+pattern/(maxint*2), xs, 'r')
    plt.show()
    print("visibility (Imax-Imin)/(Imax+Imin): ",(pattern.max()-pattern.min())/(total_intensity*2))
# show patern for one source at 0
michelson(p0=[0])


在该模型中，光源位置用波前到达角表示，基线长度以波长为单位，因而更接近远场天体产生近似平面波的观测条件。两个交互模型分别显示单源情况下可见度相位与位置的关系，以及双源情况下可见度振幅随源结构的变化。


In [ ]:
# single source
interact(lambda position, intensity, baseline: 
             michelson(p0=[position], a0=[intensity], baseline=baseline, maxint=2),
         position=(-5,5,.01),intensity=(.2,1,.01),baseline=(10,100,.01)) and None

下面将模型扩展为双光源情形：


In [ ]:
interact(lambda position1,position2,intensity1,intensity2,baseline: 
            michelson(p0=[position1,position2], a0=[intensity1,intensity2], baseline=baseline, maxint=2),
         position1=(-5,5,.01), position2=(-5,5,.01), intensity1=(.2,1,.01), intensity2=(.2,1,.01),
         baseline=(10,100,.01)) and None

#### A.1 参宿四角直径测量

本模型通过改变基线长度并记录条纹可见度的下降，重现 Michelson 和 Pease 利用干涉法测量参宿四角直径的基本原理。该示例进一步说明，可见度振幅约束源尺度，而可见度相位约束源位置。


In [ ]:
arcsec = 1/3600.
interact(lambda extent_arcsec, baseline: 
             michelson(p0=[0], a0=[1], extent=extent_arcsec*arcsec, maxint=1, 
                       baseline=baseline,fov=1*arcsec),
         extent_arcsec=(0,0.1,0.001), 
         baseline=(1e+4,1e+7,1e+4)
        ) and None

***

* 下一节： [1.10 单碟望远镜的局限与价值](1_10_limits_of_single_dishes.ipynb)
